# BERT MBTI Personality Classifier Training

This notebook trains a BERT-based classifier to predict MBTI personality types from user reviews.

In [3]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import torch
from transformers import AutoTokenizer

from src.config import get_config
from src.config.settings import PROCESSED_DATA_DIR, CHECKPOINT_DIR
from src.data.preprocessor import ReviewDataset
from src.data.sampler import DataSampler
from src.models.bert_mbti import (
    MBTIClassifier,
    MBTITrainer,
    MBTIInference,
    create_data_loaders,
)
from src.utils.helpers import set_seed, get_device

# Setup
config = get_config()
set_seed(42)
device = get_device('cuda')  # or 'mps' for Apple Silicon, 'cpu'
print(f"Using device: {device}")

Using device: cuda


## 1. Load and Prepare Data

### Option A: Use External MBTI Dataset

In [ ]:
mbti_csv_path = PROCESSED_DATA_DIR.parent / 'raw' / 'mbti_1.csv'

if mbti_csv_path.exists():
    mbti_df = pd.read_csv(mbti_csv_path)
    print(f"Loaded {len(mbti_df)} MBTI-labeled samples")
    print(f"Columns: {mbti_df.columns.tolist()}")
    
    # The Kaggle dataset has 'type' and 'posts' columns
    # 'posts' contains multiple posts separated by '|||'
    
    # Explode posts into individual rows
    mbti_df['posts_list'] = mbti_df['posts'].str.split('|||')
    mbti_exploded = mbti_df.explode('posts_list').reset_index(drop=True)
    mbti_exploded = mbti_exploded.rename(columns={'type': 'mbti', 'posts_list': 'text'})
    mbti_exploded = mbti_exploded[['mbti', 'text']]
    
    # Clean text
    from src.data.preprocessor import TextPreprocessor
    preprocessor = TextPreprocessor()
    mbti_exploded = preprocessor.process_dataframe(mbti_exploded, text_column='text')
    
    print(f"After exploding: {len(mbti_exploded)} samples")
    print(f"MBTI distribution:\n{mbti_exploded['mbti'].value_counts()}")
else:
    print(f"MBTI dataset not found at {mbti_csv_path}")
    print("Using synthetic labels for demonstration...")
    mbti_exploded = None

Loaded 8675 MBTI-labeled samples
Columns: ['type', 'posts']


In [ ]:
# Option B: Use Yelp reviews with synthetic labels (for demonstration only)

if mbti_exploded is None:
    train_path = PROCESSED_DATA_DIR / 'train_reviews.parquet'
    val_path = PROCESSED_DATA_DIR / 'val_reviews.parquet'
    
    if train_path.exists():
        train_df = pd.read_parquet(train_path)
        val_df = pd.read_parquet(val_path)
        
        # Create synthetic MBTI labels (NOT for production use)
        mbti_types = MBTIClassifier.MBTI_TYPES
        np.random.seed(42)
        train_df['mbti'] = np.random.choice(mbti_types, size=len(train_df))
        val_df['mbti'] = np.random.choice(mbti_types, size=len(val_df))
        
        print(f"Train: {len(train_df)}, Val: {len(val_df)}")
    else:
        raise FileNotFoundError("No data found. Run data preparation first.")

In [ ]:
# If using external MBTI dataset, split it
if mbti_exploded is not None:
    sampler = DataSampler()
    
    # Upsample minority classes
    mbti_balanced = sampler.upsample_minority_classes(
        mbti_exploded,
        label_column='mbti',
        target_ratio=0.5  # Balance to 50% of majority class
    )
    
    # Split
    train_df, val_df, test_df = sampler.train_val_test_split(
        mbti_balanced,
        stratify_column='mbti'
    )
    
    print(f"After balancing and splitting:")
    print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

## 2. Visualize Class Distribution

In [ ]:
# Plot MBTI distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Train distribution
train_counts = train_df['mbti'].value_counts().sort_index()
train_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Training Set MBTI Distribution')
axes[0].set_xlabel('MBTI Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Val distribution
val_counts = val_df['mbti'].value_counts().sort_index()
val_counts.plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Validation Set MBTI Distribution')
axes[1].set_xlabel('MBTI Type')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Create Datasets and DataLoaders

In [6]:
# Determine text column
text_col = 'clean_text' if 'clean_text' in train_df.columns else 'text'
print(f"Using text column: {text_col}")

# Create datasets
train_dataset = ReviewDataset.from_dataframe(
    train_df,
    text_column=text_col,
    label_column='mbti',
    config=config.bert,
)

val_dataset = ReviewDataset.from_dataframe(
    val_df,
    text_column=text_col,
    label_column='mbti',
    config=config.bert,
)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size: {len(val_dataset)}")

Using text column: clean_text


c:\Users\ewenl\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Train dataset size: 78097581
Val dataset size: 9762198


In [7]:
# Inspect a sample
sample = train_dataset[0]
print("Sample keys:", sample.keys())
print("Input IDs shape:", sample['input_ids'].shape)
print("Attention mask shape:", sample['attention_mask'].shape)
print("Label:", sample['labels'].item(), "=>", MBTIClassifier.MBTI_TYPES[sample['labels'].item()])

Sample keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Input IDs shape: torch.Size([512])
Attention mask shape: torch.Size([512])
Label: 4 => INFJ


In [8]:
# Create data loaders
BATCH_SIZE = 16  # Reduce if OOM

train_loader, val_loader = create_data_loaders(
    train_dataset,
    val_dataset,
    batch_size=BATCH_SIZE,
    num_workers=0,  # Set to 0 for notebooks
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Train batches: 4881099
Val batches: 610138


## 4. Initialize Model and Trainer

In [12]:
train_df.head(10)

,mbti,text,clean_text
22292947,INFJ,e,e
51715100,ISTP,:,:
17169574,INFP,t,t
82518569,INFJ,o,o
53455695,ISTJ,s,s
96049228,ENTJ,a,a
70406790,INFP,f,f
32473266,ISTP,t,t
52982442,INTP,P,p
76287142,INFP,u,u


In [13]:
# Create model
model = MBTIClassifier(config=config.bert)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

c:\Users\ewenl\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Total parameters: 109,494,544
Trainable parameters: 109,494,544


In [2]:
# Configure training
config.bert.num_epochs = 1  # Reduce for demo
config.bert.learning_rate = 2e-5
config.bert.early_stopping_patience = 2

# Create trainer
trainer = MBTITrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config.bert,
    device=device,
)

NameError: name 'model' is not defined

## 5. Train the Model

In [ ]:
# Train
history = trainer.train()

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
train_loss = [h['loss'] for h in history['train_history']]
val_loss = [h['loss'] for h in history['val_history']]

axes[0].plot(train_loss, label='Train', marker='o')
axes[0].plot(val_loss, label='Validation', marker='o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
train_acc = [h['accuracy'] for h in history['train_history']]
val_acc = [h['accuracy'] for h in history['val_history']]

axes[1].plot(train_acc, label='Train', marker='o')
axes[1].plot(val_acc, label='Validation', marker='o')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 6. Evaluate the Model

In [ ]:
# Print classification report
print("\nClassification Report:")
print(trainer.get_classification_report())

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']
        
        preds, _ = model.predict(input_ids, attention_mask)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    xticklabels=MBTIClassifier.MBTI_TYPES,
    yticklabels=MBTIClassifier.MBTI_TYPES,
    cmap='Blues'
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('MBTI Classification Confusion Matrix')
plt.tight_layout()
plt.show()

## 7. Test Inference

In [ ]:
# Load inference pipeline
inference = MBTIInference(
    model_path=trainer.checkpoint_dir / 'best_model.pt',
    device=device,
)

In [ ]:
# Test single prediction
test_text = """
I really enjoyed the quiet atmosphere of this cafe. It's perfect for reading 
or working on my laptop. The staff respects your privacy and doesn't bother 
you too much. I prefer places like this over loud, crowded restaurants.
"""

result = inference.predict_single(test_text)

print(f"Predicted MBTI: {result['mbti_type']}")
print(f"\nTop 5 probabilities:")
sorted_probs = sorted(result['probabilities'].items(), key=lambda x: x[1], reverse=True)
for mbti, prob in sorted_probs[:5]:
    print(f"  {mbti}: {prob:.4f}")
print(f"\nEmbedding shape: {result['embedding'].shape}")

In [ ]:
# Test batch prediction
test_texts = [
    "I love organizing events and meeting new people! The energy here is amazing.",
    "This place is perfect for deep thinking. I spent hours analyzing their menu.",
    "Great for parties! Everyone was dancing and having fun!",
    "I appreciate the logical layout and efficient service.",
]

batch_results = inference.predict_batch(test_texts, return_embeddings=True)

for i, (text, mbti) in enumerate(zip(test_texts, batch_results['mbti_types'])):
    print(f"{i+1}. {mbti}: {text[:60]}...")

## 8. Generate User MBTI Profiles

Aggregate predictions across all of a user's reviews to get their MBTI profile.

In [ ]:
# If you have user-level data with user_id
if 'user_id' in val_df.columns:
    # Sample a subset for demo
    sample_users = val_df['user_id'].unique()[:100]
    sample_df = val_df[val_df['user_id'].isin(sample_users)].copy()
    
    # Generate user MBTI profiles
    user_profiles = inference.predict_users(
        sample_df,
        user_id_col='user_id',
        text_col=text_col,
        aggregation='mean_probs',
        batch_size=32,
    )
    
    print(f"Generated profiles for {len(user_profiles)} users")
    print(user_profiles.head())
    
    # Distribution of user MBTI types
    plt.figure(figsize=(10, 5))
    user_profiles['mbti_type'].value_counts().sort_index().plot(kind='bar')
    plt.title('User MBTI Type Distribution')
    plt.xlabel('MBTI Type')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

## 9. Save User Profiles for Downstream Use

In [ ]:
# Save user profiles
if 'user_profiles' in dir():
    # Save without embeddings (too large)
    user_profiles_save = user_profiles.drop(columns=['embedding'], errors='ignore')
    user_profiles_save.to_parquet(PROCESSED_DATA_DIR / 'user_mbti_profiles.parquet', index=False)
    
    # Save embeddings separately as numpy
    embeddings = np.stack(user_profiles['embedding'].values)
    np.save(PROCESSED_DATA_DIR / 'user_mbti_embeddings.npy', embeddings)
    
    print(f"Saved user profiles to {PROCESSED_DATA_DIR}")
    print(f"Embeddings shape: {embeddings.shape}")

## Next Steps

1. **Phase 3**: Use BERTopic for venue topic modeling
2. **Phase 4**: Build heterogeneous graph with user MBTI and venue topics
3. **Phase 5**: Train hybrid recommendation model